# Revisión de evidencias del piloto Mastcam-Z

Esta segunda etapa transforma los resultados técnicos del piloto en una tabla trazable para revisión humana. Trabaja exclusivamente con las 40 imágenes locales ya seleccionadas y no descarga datos ni modelos automáticamente.

El resultado describe diferencias visuales dentro de este conjunto. No constituye evidencia de vida, fósiles, minerales ni ningún descubrimiento científico.

## Preparación y límites

El notebook espera las imágenes en `data/raw/mastcam_z/pilot/` y el manifiesto en `data/metadata/mastcam_z_pilot.csv`. Los embeddings se reutilizan si existen; si faltan, la celda explícita de DINOv2 los genera y guarda.

La similitud y la anomalía se calculan en el espacio original de embeddings. UMAP se utiliza únicamente para visualizar relaciones, no como una distancia científica exacta.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image

from mars_evidence.analysis.anomaly import visual_anomaly_scores
from mars_evidence.analysis.clustering import cluster_hdbscan, reduce_umap
from mars_evidence.analysis.evidence import (
    build_pilot_evidence_table,
    deterministic_image_order,
    export_evidence_table,
    load_embedding_artifact,
    nearest_neighbors_table,
    save_embedding_artifacts,
    validate_embedding_alignment,
)
from mars_evidence.config import get_device, load_config, set_seed
from mars_evidence.data.manifest import duplicate_groups
from mars_evidence.data.validation import validate_manifest
from mars_evidence.features.dinov2 import DINOv2Embedder

sns.set_theme(style="whitegrid")

In [ ]:
ROOT = Path.cwd()
CONFIG_PATH = ROOT / "config" / "base.yaml"
CONFIG = load_config(CONFIG_PATH)
set_seed(int(CONFIG["project"]["seed"]))

RAW_DIR = ROOT / "data" / "raw" / "mastcam_z" / "pilot"
MANIFEST_PATH = ROOT / "data" / "metadata" / "mastcam_z_pilot.csv"
EMBEDDING_PATH = ROOT / "outputs" / "embeddings" / "dinov2_small_pilot.npy"
ORDER_PATH = ROOT / "outputs" / "embeddings" / "dinov2_small_pilot_image_ids.npy"
LEGACY_NPZ_PATH = ROOT / "outputs" / "embeddings" / "dinov2_embeddings.npz"
FIGURES_DIR = ROOT / "outputs" / "figures"
REPORTS_DIR = ROOT / "outputs" / "reports"

READY = RAW_DIR.exists() and MANIFEST_PATH.exists()
print(f"Dispositivo disponible: {get_device()}")
print(f"Dataset listo: {READY}")
if not READY:
    print(
        "Copia las imágenes en data/raw/mastcam_z/pilot/ y el manifiesto en "
        "data/metadata/mastcam_z_pilot.csv antes de continuar."
    )

## Carga del manifiesto y resumen del conjunto

El orden de trabajo se fija de forma determinista por `image_id`. Los metadatos se muestran tal como aparecen en el manifiesto; los valores ausentes permanecen vacíos.

In [ ]:
if READY:
    manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")
    ordered_manifest = deterministic_image_order(manifest)
    print(f"Imágenes en el manifiesto: {len(ordered_manifest)}")
    display(ordered_manifest.head())
else:
    manifest = pd.DataFrame()
    ordered_manifest = pd.DataFrame()

In [ ]:
if READY:
    validation = validate_manifest(ordered_manifest, RAW_DIR)
    print(f"Archivos legibles: {validation['readable'].sum()} / {len(validation)}")
    display(validation[~validation['readable']])
    duplicate_rows = duplicate_groups(ordered_manifest)
    print(f"Filas con SHA-256 duplicado: {len(duplicate_rows)}")
    if not duplicate_rows.empty:
        display(duplicate_rows[["image_id", "sha256"]])
else:
    validation = pd.DataFrame()
    duplicate_rows = pd.DataFrame()

In [ ]:
if READY:
    summary = pd.DataFrame(
        {
            "metric": [
                "image_count",
                "sol_count",
                "camera_count",
                "left_eye_count",
                "right_eye_count",
                "missing_source_url",
                "missing_earth_date",
                "missing_camera",
            ],
            "value": [
                len(ordered_manifest),
                ordered_manifest["sol"].replace("", np.nan).nunique(),
                ordered_manifest["camera"].replace("", np.nan).nunique(),
                (ordered_manifest["eye"] == "left").sum(),
                (ordered_manifest["eye"] == "right").sum(),
                ordered_manifest["source_url"].eq("").sum(),
                ordered_manifest["earth_date"].eq("").sum(),
                ordered_manifest["camera"].eq("").sum(),
            ],
        }
    )
    display(summary)
    display(
        ordered_manifest.groupby(["width", "height", "format"], dropna=False)
        .size()
        .reset_index(name="image_count")
    )

## Embeddings: reutilización, alineación y generación explícita

Se conserva un archivo separado con el orden exacto de `image_id`. Si se encuentra el NPZ legado del piloto, se migra localmente a los archivos `.npy` de esta etapa. Si no hay ningún artefacto, ejecuta esta celda: la carga de DINOv2 es deliberada y puede tardar, especialmente en CPU.

In [ ]:
if READY:
    embeddings, embedding_ids, embeddings_available = load_embedding_artifact(
        EMBEDDING_PATH,
        ORDER_PATH,
        LEGACY_NPZ_PATH,
    )
    if embeddings_available:
        id_to_row = {image_id: index for index, image_id in enumerate(embedding_ids)}
        expected_ids = ordered_manifest["image_id"].astype(str).tolist()
        if set(embedding_ids) != set(expected_ids):
            raise ValueError("Los embeddings no corresponden al conjunto de imágenes del manifiesto.")
        embeddings = np.asarray(
            [embeddings[id_to_row[image_id]] for image_id in expected_ids],
            dtype=np.float32,
        )
        save_embedding_artifacts(
            embeddings,
            expected_ids,
            EMBEDDING_PATH,
            ORDER_PATH,
        )
        embedding_ids = np.asarray(expected_ids)
        print("Embeddings reutilizados y alineados con el orden determinista.")
    else:
        print("No hay embeddings locales. Ejecuta esta celda para generar DINOv2.")
        embedder = DINOv2Embedder(model_name=CONFIG["model"]["name"])
        image_paths = [RAW_DIR / path for path in ordered_manifest["relative_path"]]
        embeddings = embedder.embed_images(
            image_paths,
            batch_size=int(CONFIG["model"]["batch_size"]),
        )
        embedding_ids = ordered_manifest["image_id"].astype(str).to_numpy()
        save_embedding_artifacts(
            embeddings,
            embedding_ids,
            EMBEDDING_PATH,
            ORDER_PATH,
        )
    validate_embedding_alignment(
        embedding_ids,
        ordered_manifest["image_id"].astype(str),
    )
    print(f"Dimensión de embeddings: {embeddings.shape}")
else:
    embeddings = np.empty((0, 0), dtype=np.float32)
    embedding_ids = np.asarray([], dtype=str)

## UMAP, HDBSCAN y anomalía visual

HDBSCAN se ajusta en el espacio de embeddings. UMAP solo produce una proyección para inspección visual. La puntuación de anomalía es una distancia coseno media a vecinos cercanos y se normaliza entre 0 y 1; no es una probabilidad científica.

In [ ]:
if READY and len(embeddings):
    umap_projection = reduce_umap(
        embeddings,
        n_neighbors=int(CONFIG["umap"]["n_neighbors"]),
        min_dist=float(CONFIG["umap"]["min_dist"]),
        n_components=int(CONFIG["umap"]["n_components"]),
        metric=str(CONFIG["umap"]["metric"]),
        random_state=int(CONFIG["project"]["seed"]),
    )
    cluster_labels = cluster_hdbscan(
        embeddings,
        min_cluster_size=int(CONFIG["hdbscan"]["min_cluster_size"]),
        min_samples=int(CONFIG["hdbscan"]["min_samples"]),
    )
    anomaly_scores = visual_anomaly_scores(
        embeddings,
        neighbors=int(CONFIG["anomaly"]["neighbors"]),
    )
    neighbors = nearest_neighbors_table(
        embeddings,
        ordered_manifest["image_id"].astype(str),
        k=3,
    )
    print(f"Clústeres: {len(set(cluster_labels)) - int(-1 in cluster_labels)}")
    print(f"Imágenes clasificadas como ruido: {(cluster_labels == -1).sum()}")
else:
    umap_projection = np.empty((0, 2))
    cluster_labels = np.asarray([], dtype=int)
    anomaly_scores = np.asarray([], dtype=float)
    neighbors = pd.DataFrame()

## Tabla de evidencias y exportación

La tabla combina únicamente metadatos del manifiesto con métricas calculadas. Las columnas de revisión humana se inicializan sin conclusiones.

In [ ]:
if READY and len(embeddings):
    evidence_table = build_pilot_evidence_table(
        ordered_manifest,
        cluster_labels,
        anomaly_scores,
        neighbors,
    )
    evidence_table["reviewed"] = False
    evidence_table["expected_terrain"] = ""
    evidence_table["rover_artifact"] = False
    evidence_table["image_quality_issue"] = False
    evidence_table["requires_further_review"] = False
    export_evidence_table(
        evidence_table,
        REPORTS_DIR / "pilot_evidence_table.csv",
    )
    top_anomalies = evidence_table.head(10).copy()
    export_evidence_table(top_anomalies, REPORTS_DIR / "top_anomalies.csv")
    display(top_anomalies)
else:
    evidence_table = pd.DataFrame()
    top_anomalies = pd.DataFrame()

In [ ]:
if READY and len(embeddings):
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    figure, axis = plt.subplots(figsize=(10, 7))
    scatter = axis.scatter(
        umap_projection[:, 0],
        umap_projection[:, 1],
        c=cluster_labels,
        cmap="tab20",
        s=55,
    )
    axis.set_title("UMAP del piloto Mastcam-Z por clúster HDBSCAN")
    axis.set_xlabel("UMAP-1")
    axis.set_ylabel("UMAP-2")
    figure.colorbar(scatter, ax=axis, label="cluster_id")
    figure.tight_layout()
    figure.savefig(FIGURES_DIR / "pilot_umap_clusters.png", dpi=160)
    plt.show()
    plt.close(figure)

## Diez imágenes con mayor puntuación de anomalía

Estas imágenes son candidatas a revisión por diferencia visual relativa al conjunto. No son anomalías científicas.

In [ ]:
def show_grid(records: pd.DataFrame, output_path: Path, title: str) -> None:
    columns = 4
    rows = max(1, (len(records) + columns - 1) // columns)
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 4 * rows), squeeze=False)
    for position, record in enumerate(records.to_dict("records")):
        axis = axes.flat[position]
        path = RAW_DIR / str(record["relative_path"])
        try:
            with Image.open(path) as image:
                axis.imshow(image.convert("RGB"))
            axis.set_title(
                f'{record["image_id"]}\nscore={record["anomaly_score"]:.3f}',
                fontsize=8,
            )
        except (OSError, ValueError) as exc:
            axis.text(0.5, 0.5, f"No legible\n{exc}", ha="center", va="center")
        axis.axis("off")
    for axis in axes.flat[len(records):]:
        axis.axis("off")
    figure.suptitle(title)
    figure.tight_layout()
    figure.savefig(output_path, dpi=160)
    plt.show()
    plt.close(figure)

if READY and not top_anomalies.empty:
    show_grid(
        top_anomalies,
        FIGURES_DIR / "top_anomalies_grid.png",
        "Top 10 de diferencia visual relativa",
    )
    display(top_anomalies[["image_id", "relative_path", "anomaly_score", "anomaly_rank"]])

## Vecinos visuales de las cinco primeras anomalías

Los vecinos se calculan con similitud coseno en el espacio original de embeddings. Para cada consulta se muestran tres vecinos, sus puntuaciones y los metadatos disponibles.

In [ ]:
if READY and not top_anomalies.empty:
    neighbor_records = []
    detail_rows = []
    manifest_by_id = ordered_manifest.set_index("image_id")
    for record in top_anomalies.head(5).to_dict("records"):
        query_id = record["image_id"]
        query_path = RAW_DIR / str(record["relative_path"])
        neighbor_records.append(
            {
                "image_id": query_id,
                "relative_path": record["relative_path"],
                "anomaly_score": record["anomaly_score"],
            }
        )
        for rank in range(1, 4):
            neighbor_id = record[f"nearest_image_{rank}"]
            neighbor = manifest_by_id.loc[neighbor_id]
            neighbor_records.append(
                {
                    "image_id": neighbor_id,
                    "relative_path": neighbor["relative_path"],
                    "anomaly_score": np.nan,
                }
            )
            detail_rows.append(
                {
                    "query_image": query_id,
                    "neighbor_rank": rank,
                    "neighbor_image": neighbor_id,
                    "similarity": record[f"nearest_similarity_{rank}"],
                    "path": neighbor["relative_path"],
                    "sol": neighbor.get("sol", ""),
                    "camera": neighbor.get("camera", ""),
                    "earth_date": neighbor.get("earth_date", ""),
                }
            )
    neighbor_frame = pd.DataFrame(neighbor_records)
    neighbor_frame["display_score"] = neighbor_frame["anomaly_score"].fillna(-1)
    show_grid(
        neighbor_frame,
        FIGURES_DIR / "anomaly_neighbors_grid.png",
        "Consulta y tres vecinos visuales por anomalía",
    )
    display(pd.DataFrame(detail_rows))
else:
    detail_rows = []

## Interfaz de revisión humana

`review_template` es un DataFrame editable. Completa sus columnas sin alterar los metadatos oficiales. Los valores iniciales no expresan una decisión sobre ninguna imagen.

In [ ]:
if READY and not evidence_table.empty:
    review_columns = [
        "image_id",
        "human_review_status",
        "human_explanation",
        "possible_artifact",
        "review_notes",
        "reviewed",
        "expected_terrain",
        "rover_artifact",
        "image_quality_issue",
        "requires_further_review",
    ]
    review_template = evidence_table[review_columns].copy()
    display(review_template.head(10))
    print(
        "Edita review_template y exporta de nuevo evidence_table cuando termine "
        "la revisión humana."
    )
else:
    review_template = pd.DataFrame()

## Cómo interpretar los resultados

Anomalía significa diferencia visual dentro de este conjunto de imágenes, según la distancia a sus vecinos en el espacio de embeddings. No significa objeto desconocido, evidencia biológica ni descubrimiento científico.

Una puntuación alta puede deberse a iluminación, compresión, desenfoque, una parte del rover, un terreno diferente o la baja diversidad del dataset. Toda imagen señalada requiere revisión humana y, si procede, comparación con sus metadatos y productos oficiales.

## Conclusión automática del piloto

La siguiente conclusión se genera exclusivamente a partir de las métricas calculadas. No interpreta el contenido de las imágenes como evidencia científica.

In [ ]:
if READY and not evidence_table.empty:
    cluster_count = evidence_table.loc[
        ~evidence_table["is_cluster_noise"], "cluster_id"
    ].nunique()
    noise_count = int(evidence_table["is_cluster_noise"].sum())
    score_min = float(evidence_table["anomaly_score"].min())
    score_max = float(evidence_table["anomaly_score"].max())
    conclusion = (
        f"El piloto contiene {len(evidence_table)} imágenes. "
        f"HDBSCAN produjo {cluster_count} clústeres y {noise_count} imágenes marcadas "
        f"como ruido. El rango de puntuaciones visuales es "
        f"{score_min:.4f}–{score_max:.4f}. "
        "Estas métricas dependen del modelo, los parámetros y la diversidad del piloto; "
        "la compresión, la iluminación, el desenfoque, partes del rover y la cobertura "
        "limitada del conjunto pueden influir en los resultados. Se requiere revisión "
        "humana antes de cualquier interpretación adicional."
    )
    print(conclusion)
else:
    print(
        "No se puede generar la conclusión: falta el manifiesto, las imágenes o los "
        "embeddings alineados."
    )

## Limitaciones y próximos pasos

Este piloto técnico no representa toda la diversidad de Mastcam-Z y no establece una línea base científica. Antes de ampliar el análisis conviene revisar manualmente las imágenes señaladas, documentar las decisiones de revisión y controlar mejor la composición del dataset.

No se implementan todavía PatchCore, visión estéreo ni análisis químico. Tampoco se usa un LLM para interpretar resultados científicos.